# Bot 1 Trading Behavior Analysis

Reverse engineer the trading patterns of Bot 1 in VEV (voucher) and VFE (underlying) products

In [ ]:
import json
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Load submission log
log_path = 'Submission logs/544821/544821.json'
print(f'Loading {log_path}...')

with open(log_path, 'r') as f:
    lines = f.readlines()

print(f'Loaded {len(lines)} lines')
print(f'First line sample: {lines[0][:200]}')

In [ ]:
# Parse JSON lines into structured data
logs = []
for i, line in enumerate(lines):
    try:
        log_entry = json.loads(line.strip())
        logs.append(log_entry)
    except json.JSONDecodeError as e:
        if i < 5:
            print(f'Error on line {i}: {e}')

print(f'Successfully parsed {len(logs)} log entries')
print(f'\nSample log entry keys:')
if logs:
    print(logs[0].keys())

In [ ]:
# Extract market trades from logs (these contain Bot 1 activity)
bot1_trades = []

for iteration, log_entry in enumerate(logs):
    if 'market_trades' not in log_entry:
        continue
    
    market_trades_dict = log_entry['market_trades']
    
    for product, trades_list in market_trades_dict.items():
        for trade in trades_list:
            trade_record = {
                'iteration': iteration,
                'timestamp': log_entry.get('timestamp', 0),
                'product': product,
                'price': trade.get('price'),
                'quantity': trade.get('quantity'),
                'buyer': trade.get('buyer'),
                'seller': trade.get('seller'),
            }
            
            # Filter for Bot 1 trades (any market trade where we're not involved)
            # Bot 1 is identifiable by consistent trading pattern
            if trade_record['buyer'] is None or trade_record['buyer'] == '':
                bot1_trades.append(trade_record)

df_bot1 = pd.DataFrame(bot1_trades)
print(f'Total Bot 1 trades: {len(df_bot1)}')
print(f'\nTrades by product:')
print(df_bot1['product'].value_counts())

In [ ]:
# Filter for VFE and VEV products
df_vfe_vev = df_bot1[df_bot1['product'].isin(['VELVETFRUIT_EXTRACT'] + 
                      [f'VEV_{strike}' for strike in [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500]])].copy()

print(f'VFE + VEV trades: {len(df_vfe_vev)}')
print(f'\nBreakdown:')
print(df_vfe_vev['product'].value_counts().sort_index())

In [ ]:
# Analyze VFE trading pattern
vfe_trades = df_vfe_vev[df_vfe_vev['product'] == 'VELVETFRUIT_EXTRACT'].copy()

print('VFE Trading Statistics:')
print(f'Total trades: {len(vfe_trades)}')
print(f'Price range: {vfe_trades["price"].min()} - {vfe_trades["price"].max()}')
print(f'Mean price: {vfe_trades["price"].mean():.2f}')
print(f'Median price: {vfe_trades["price"].median():.2f}')
print(f'\nQuantity statistics:')
print(vfe_trades['quantity'].describe())

# Calculate price spread/volatility
print(f'\nPrice volatility (std): {vfe_trades["price"].std():.2f}')
print(f'Price changes (consecutive trades):')
price_changes = vfe_trades['price'].diff()
print(f'  Mean change: {price_changes.mean():.2f}')
print(f'  Std change: {price_changes.std():.2f}')

In [ ]:
# Analyze VEV trading patterns by strike
vev_trades = df_vfe_vev[df_vfe_vev['product'].str.startswith('VEV')].copy()

# Extract strike from product name
vev_trades['strike'] = vev_trades['product'].str.extract(r'(\d+)').astype(int)

print('VEV Trading Statistics by Strike:')
vev_summary = vev_trades.groupby('strike').agg({
    'quantity': ['count', 'mean', 'sum'],
    'price': ['min', 'max', 'mean', 'std']
}).round(2)
print(vev_summary)

In [ ]:
# Temporal analysis: Are there patterns in when Bot 1 trades?
vfe_trades = vfe_trades.sort_values('iteration')
vfe_trades['iteration_diff'] = vfe_trades['iteration'].diff()

print('VFE Trading Frequency Analysis:')
print(f'Iteration span: {vfe_trades["iteration"].min()} to {vfe_trades["iteration"].max()}')
print(f'Total iterations: {vfe_trades["iteration"].max() - vfe_trades["iteration"].min()}')
print(f'Trade frequency (trades per 100 iterations): {(len(vfe_trades) / (vfe_trades["iteration"].max() - vfe_trades["iteration"].min())) * 100:.2f}')

print(f'\nGaps between consecutive trades (iterations):')
print(f'  Mean gap: {vfe_trades["iteration_diff"].mean():.2f}')
print(f'  Median gap: {vfe_trades["iteration_diff"].median():.2f}')
print(f'  Min gap: {vfe_trades["iteration_diff"].min():.0f}')
print(f'  Max gap: {vfe_trades["iteration_diff"].max():.0f}')

In [ ]:
# Quantity pattern analysis: Is Bot 1 sizing consistently?
print('VFE Quantity Pattern Analysis:')
quantity_counts = vfe_trades['quantity'].value_counts().head(10)
print(quantity_counts)
print(f'\nMost common quantities (% of trades):')
print((quantity_counts / len(vfe_trades) * 100).round(2))

In [ ]:
# Detect directionality: Is Bot 1 more buyer or seller?
# Check through own_trades - when we're buyer, bot is seller (negative), vice versa

# Extract our trades from logs
our_trades_list = []
for iteration, log_entry in enumerate(logs):
    if 'own_trades' not in log_entry:
        continue
    
    own_trades_dict = log_entry['own_trades']
    
    for product, trades_list in own_trades_dict.items():
        for trade in trades_list:
            if product in ['VELVETFRUIT_EXTRACT'] + [f'VEV_{s}' for s in [4000, 4500, 5000, 5100, 5200, 5300, 5400, 5500, 6000, 6500]]:
                our_trade_record = {
                    'iteration': iteration,
                    'product': product,
                    'price': trade.get('price'),
                    'quantity': trade.get('quantity'),
                    'buyer': trade.get('buyer'),
                    'seller': trade.get('seller'),
                }
                # If we're buyer (positive quantity), Bot 1 is seller
                # If we're seller (negative quantity), Bot 1 is buyer
                our_trades_list.append(our_trade_record)

df_our_trades = pd.DataFrame(our_trades_list)

print(f'Our VFE + VEV trades: {len(df_our_trades)}')
print(f'\nBreakdown by product:')
print(df_our_trades['product'].value_counts())

In [ ]:
# Analyze our trading against Bot 1 - infer Bot 1's strategy
vfe_our_trades = df_our_trades[df_our_trades['product'] == 'VELVETFRUIT_EXTRACT'].copy()

print('Our VFE Trades (Interaction with Bot 1):')
print(f'Total trades: {len(vfe_our_trades)}')

# Determine Bot 1 side
vfe_our_trades['we_are_buyer'] = vfe_our_trades['quantity'] > 0
vfe_our_trades['bot1_side'] = vfe_our_trades['we_are_buyer'].apply(lambda x: 'SELLER' if x else 'BUYER')

print(f'\nBot 1 activity in our trades:')
print(vfe_our_trades['bot1_side'].value_counts())

print(f'\nPrices when Bot 1 sells to us:')
bot1_seller = vfe_our_trades[vfe_our_trades['bot1_side'] == 'SELLER']
print(f'  Count: {len(bot1_seller)}')
print(f'  Avg price: {bot1_seller["price"].mean():.2f}')
print(f'  Price range: {bot1_seller["price"].min()}-{bot1_seller["price"].max()}')

print(f'\nPrices when Bot 1 buys from us:')
bot1_buyer = vfe_our_trades[vfe_our_trades['bot1_side'] == 'BUYER']
print(f'  Count: {len(bot1_buyer)}')
print(f'  Avg price: {bot1_buyer["price"].mean():.2f}')
print(f'  Price range: {bot1_buyer["price"].min()}-{bot1_buyer["price"].max()}')

In [ ]:
# Analyze VEV patterns - does Bot 1 show preference for certain strikes?
vev_our_trades = df_our_trades[df_our_trades['product'].str.startswith('VEV')].copy()
vev_our_trades['strike'] = vev_our_trades['product'].str.extract(r'(\d+)').astype(int)
vev_our_trades['we_are_buyer'] = vev_our_trades['quantity'] > 0

print('VEV Trading Against Bot 1 by Strike:')
vev_summary = vev_our_trades.groupby('strike').agg({
    'quantity': 'count',
    'price': ['mean', 'min', 'max'],
}).round(2)
vev_summary.columns = ['trade_count', 'avg_price', 'min_price', 'max_price']
print(vev_summary.sort_index())

In [ ]:
# Determine Bot 1 bias per VEV strike
print('Bot 1 Buying/Selling Bias by VEV Strike:')
vev_our_trades['bot1_side'] = vev_our_trades['we_are_buyer'].apply(lambda x: 'SELLER' if x else 'BUYER')

for strike in sorted(vev_our_trades['strike'].unique()):
    strike_data = vev_our_trades[vev_our_trades['strike'] == strike]
    if len(strike_data) == 0:
        continue
    
    n_seller = (strike_data['bot1_side'] == 'SELLER').sum()
    n_buyer = (strike_data['bot1_side'] == 'BUYER').sum()
    total = len(strike_data)
    
    print(f'VEV_{strike}: Seller {n_seller}x ({n_seller/total*100:.0f}%), Buyer {n_buyer}x ({n_buyer/total*100:.0f}%) | Avg price: {strike_data["price"].mean():.2f}')

In [ ]:
# Plot VFE price over time
vfe_timeline = vfe_trades.sort_values('iteration')

plt.figure(figsize=(14, 6))
plt.subplot(1, 2, 1)
plt.scatter(vfe_timeline['iteration'], vfe_timeline['price'], alpha=0.6, s=vfe_timeline['quantity']/10)
plt.xlabel('Iteration')
plt.ylabel('Price')
plt.title('Bot 1 VFE Trading - Price Over Time (size = quantity)')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(vfe_timeline['price'], bins=50, edgecolor='black', alpha=0.7)
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.title('VFE Price Distribution')
plt.tight_layout()
plt.show()

In [ ]:
# VEV Strike preference visualization
vev_strike_counts = vev_our_trades['strike'].value_counts()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(vev_strike_counts.index, vev_strike_counts.values, color='steelblue', edgecolor='black')
ax1.set_xlabel('Strike')
ax1.set_ylabel('Trade Count')
ax1.set_title('Bot 1 VEV Trading Activity by Strike')
ax1.grid(True, alpha=0.3, axis='y')

# Buyer/Seller breakdown
vev_side_breakdown = vev_our_trades.groupby(['strike', 'bot1_side']).size().unstack(fill_value=0)
vev_side_breakdown.plot(kind='bar', ax=ax2, color=['red', 'green'], edgecolor='black')
ax2.set_xlabel('Strike')
ax2.set_ylabel('Trade Count')
ax2.set_title('Bot 1 Buy/Sell Behavior by VEV Strike')
ax2.legend(['Buyer', 'Seller'])
ax2.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

In [ ]:
# Summary insights
print('\n=== BOT 1 TRADING STRATEGY SUMMARY ===')
print(f'\n1. VFE (Underlying) Behavior:')
print(f'   - Total trades: {len(vfe_trades)}')
print(f'   - Active in {len(vfe_trades["iteration"].unique())} iterations')
print(f'   - Avg gap between trades: {vfe_trades["iteration_diff"].mean():.0f} iterations')
print(f'   - Price focus: {vfe_trades["price"].median():.0f} (median)')
print(f'   - Preferred quantity: {vfe_trades["quantity"].mode().values[0] if len(vfe_trades["quantity"].mode()) > 0 else "N/A"}')

print(f'\n2. VEV (Options) Behavior:')
print(f'   - Most active strikes: {vev_strike_counts.head(3).index.tolist()}')
print(f'   - Total VEV trades: {len(vev_our_trades)}')

if len(bot1_seller) > 0:
    print(f'\n3. Bot 1 as Seller (we buy from):') 
    print(f'   - {len(bot1_seller)} trades')
    print(f'   - Avg sell price: {bot1_seller["price"].mean():.2f}')
    print(f'   - Sell price range: {bot1_seller["price"].min()}-{bot1_seller["price"].max()}')

if len(bot1_buyer) > 0:
    print(f'\n4. Bot 1 as Buyer (bot buys from us):')
    print(f'   - {len(bot1_buyer)} trades')
    print(f'   - Avg buy price: {bot1_buyer["price"].mean():.2f}')
    print(f'   - Buy price range: {bot1_buyer["price"].min()}-{bot1_buyer["price"].max()}')